# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

# Personalized Technical Tutor

This tool takes a technical question and generates an easy to understand explanation using:

1. GPT-5-nano through OpenAI API
2. Open Source Model running locally through Ollama

Both models receive the same question and same tutor instructions.

In [49]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI


## Constants

It Stores the model names and local ollama endpoint.

GPT-5-nano will be called through OpenAI.

The open source model will be called locally through Ollama.

In [50]:
# constants

MODEL_GPT = 'gpt-5-nano'
MODEL_LLAMA = 'llama3.2'


## Set Up the Enviornment

The OpenAI API Key is loaded from the .env file.

OpenAI client connects to GPT-5-nano.

Ollama connects to Open source model running locally.

In [51]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    

openai = OpenAI()

OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama"
)

API key looks good so far


In [4]:
# here is the question; type over this to ask something new

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}

Please Explain:
- what is set comprehension does
- what book.get("author") does
- why the if condition is used
- what yield from does
- How duplicate authors are removed
- Why the order of the results may not be predictable.
"""

In [5]:
# New Question

question = """
A binary classification model produced the following confusion matrix:

- True Positives = 40
- False Positives = 10
- True Negatives = 45
- False Negatives = 5

Please calculate and explain:

- Accuracy
- Precision
- Recall
- F1-score

Show each formula, substitute the values, and give the final answer rounded to 2 decimal places.
"""

## Messages

The same System Prompt and Technical question wil be sent to both models.

This makes it easier to compare the explanation.

In [6]:
# Mesaages
messages = [
    {"role":"system", "content":"system_prompt"},
    {"role":"user", "content":question}
]

In [7]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

Given: TP=40, FP=10, TN=45, FN=5

- Accuracy
  - Formula: Accuracy = (TP + TN) / (TP + FP + TN + FN)
  - Substitution: (40 + 45) / (40 + 10 + 45 + 5) = 85 / 100 = 0.85
  - Result (2 decimals): 0.85

- Precision
  - Formula: Precision = TP / (TP + FP)
  - Substitution: 40 / (40 + 10) = 40 / 50 = 0.80
  - Result (2 decimals): 0.80

- Recall
  - Formula: Recall = TP / (TP + FN)
  - Substitution: 40 / (40 + 5) = 40 / 45 ≈ 0.8889
  - Result (2 decimals): 0.89

- F1-score
  - Formula: F1 = 2 * TP / (2*TP + FP + FN)
  - Substitution: 2*40 / (80 + 10 + 5) = 80 / 95 ≈ 0.8421
  - Result (2 decimals): 0.84

Summary: Accuracy 0.85, Precision 0.80, Recall 0.89, F1-score 0.84.

In [8]:
# Warm up Llama 3.2 - It was taking a lot of time to run, just for warm up

warmup = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=[
        {"role":"user", "content":"Hi"}
    ]
)

print("Llama 3.2 is ready")

Llama 3.2 is ready


In [9]:
# Get Llama 3.2 to answer
response = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages
)

display(Markdown(response.choices[0].message.content))

**Calculating Metrics from a Confusion Matrix**

A confusion matrix is a table used to evaluate the performance of a classification model. Here, we'll use the given values to calculate:

1. Accuracy
2. Precision
3. Recall
4. F1-score

**Accuracy**

Accuracy measures the proportion of correctly classified instances. It is calculated using the following formula:

Accuracy = (TP + TN) / (TP + TN + FP + FN)

where TP = True Positives, TN = True Negatives, FP = False Positives, and FN = False Negatives

Plugging in the values, we get:

Accuracy = (40 + 45) / (40 + 45 + 10 + 5)
= 85/80
≈ 0.53 (rounded to 2 decimal places)

**Precision**

Precision measures the proportion of predicted positive instances that are actually true positives. It is calculated using the following formula:

Precision = TP / (TP + FP)

Substituting the values, we get:

Precision = 40 / (40 + 10)
= 4/5
≈ 0.80 (rounded to 2 decimal places)

**Recall**

Recall measures the proportion of actual positive instances that are correctly predicted. It is calculated using the following formula:

Recall = TP / (TP + FN)

Substituting the values, we get:

Recall = 40 / (40 + 5)
= 4/5
≈ 0.80 (rounded to 2 decimal places)

**F1-score**

The F1-score is the weighted average of precision and recall. It is calculated using the following formula:

F1-score = 2 \* (Precision \* Recall) / (Precision + Recall)

Substituting the values, we get:

F1-score = 2 \* (0.80 \* 0.80) / (0.80 + 0.80)
= 2 \* 0.64 / 1.60
≈ 0.40 (rounded to 2 decimal places)

In summary, the performance of the binary classification model is:

* Accuracy: 53.00%
* Precision: 80.00%
* Recall: 80.00%
* F1-score: 40.00%

# Week 2 Upgrade - Conversation Context

The Personalized Technical Tutor is now improved by giving it the previous conversations as background information.

This helps the tutor understand the context before answering the new question.

## Previous Conversations

The last two conversations are stored as background information.

The tutor will use them before answering the new technical question.

In [10]:
# Previous Conversations

previous_conversation_1 = """
User: What is supervised learning?
Tutor: Supervised learning is a type of machine learning where a model learns from labeled data.
"""

previous_conversation_2 = """
User: What is overfitting?
Tutor: Overfitting happens when a model learns the training data too closely and does not perform well on new data.
"""

In [11]:
# New Question: Now a new technical question is asked based on the previous conversations.
question = """
How can I prevent overfitting when training a supervised machine learning model?

Please Explain:

- Why overfitting happens
- How train and validation data helps
- How regularization helps
- How cross validation helps
- How early stopping helps
"""

## User Prompt

The previous conversations and the new technical question are combined into one User Prompt.

This gives the tutor the conversation context before answering.

In [12]:
# User Prompt

user_prompt = f"""
Previous Conversation 1:
{previous_conversation_1}

Previous Conversation 2:
{previous_conversation_2}

New Technical Question:
{question}
"""

## System Prompt

The System Prompt defines how the Technical Tutor should explain the questions.

The same instructions will be used for both models.

In [13]:
system_prompt = """
You are a technical tutor for Data Science and AI.

Explain technical concepts in simple and clear steps.

Assume familiarity with Python, statistics, data analysis and machine learning.

When helpful:
- Explain the main idea first
- Break the answer into simple steps
- Use practical examples
- Explain formulas and calculations clearly
"""

## Messages

The same System Prompt and User Prompt will be sent to both models.

The User Prompt now contains the previous conversations and the new technical question.

In [14]:
# Messages

messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt}
]

In [15]:
# Get gpt-5-nano to answer, with streaming
stream = openai.chat.completions.create(
    model=MODEL_GPT,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(Markdown(response),
    display_id = display_handle.display_id
    )

Here’s a clear, practical guide to prevent overfitting in supervised learning. I’ll cover each of the requested points, with simple ideas you can apply step by step.

Main idea
- Overfitting happens when a model learns the training data too well (including noise) and fails to generalize to new data. The cure is to measure generalization on unseen data and to restrain model complexity or guide the learning process so it focuses on real patterns rather than noise.

1) Why overfitting happens
- Model capacity too high for the amount/signal of data: very complex models (big neural nets, deep trees) can memorize training data.
- Noisy or unrepresentative training data: if data includes noise, a flexible model may fit that noise.
- Too little data relative to features: with many features, the model can find spurious relationships that don’t hold in new data.
- All of this shows up as: low training error but high validation/test error (high variance).

2) How train and validation data helps
- Train set: used to fit the model.
- Validation set: used to monitor generalization during training and hyperparameter tuning.
- Test set (optional here): used only once, after you’ve settled on a model, to get an unbiased estimate of real-world performance.
- Practical rule: never use the test data to tune anything. Use train/validation (and possibly cross-validation) for that.
- What to look for:
  - If training error is low but validation error is high or worsens as you train, you’re likely overfitting.
  - Use validation performance to pick hyperparameters (e.g., complexity controls, learning rate, regularization strength).

3) How regularization helps
- Idea: add a penalty to the learning objective that constrains model complexity.
- Common penalties:
  - L2 regularization (ridge): adds lambda * ||w||^2 to the loss. Encourages small, spread-out weights.
  - L1 regularization (lasso): adds lambda * ||w||_1. Encourages sparsity (many weights become zero).
  - Elastic net: combination of L1 and L2.
- Effect: reduces model flexibility a bit, discourages fitting noise, tends to improve generalization.
- Typical use cases:
  - Linear models: Ridge (L2) or Lasso (L1).
  - Neural networks: weight decay (L2) and sometimes dropout (randomly drop units during training).
- How to choose strength:
  - Start with a moderate regularization strength (e.g., alpha in Ridge/Lasso) and tune it with cross-validation.

4) How cross-validation helps
- What it is: split data into K folds; train on K-1 folds and validate on the remaining fold; repeat for all folds; average results.
- Why it helps:
  - Reduces variance in performance estimates compared to a single train/validation split.
  - Gives a more reliable signal for hyperparameter tuning (e.g., regularization strength, model depth).
  - Useful when you have limited data and want to make efficient use of it.
- Practical notes:
  - Use K-fold CV for hyperparameter search (grid or random search).
  - For time-series or ordered data, use time-series cross-validation (no random shuffling).
  - Be mindful of leakage: scale/transform features using statistics from the training folds only.

5) How early stopping helps
- What it is: during iterative training, stop when validation performance stops improving (or gets worse).
- Why it helps:
  - Prevents the model from continuing to fit noise in the training data.
  - Often acts as a robust regularization technique, especially for neural networks.
- How to implement:
  - Split a validation set (or use a held-out portion of training data) and monitor a metric (e.g., validation loss or accuracy).
  - Stop after a patience period with no improvement, and restore the weights from the best observed validation performance.
- Practical tips:
  - Use a patience value (e.g., 5–10 epochs) so you don’t stop too early.
  - Saving/restoring best weights prevents a later drop in performance from the final training step.

Putting it together: a practical plan
- Step 1: Split data
  - Simple case: train/validation/test splits (e.g., 70/15/15).
  - Or use K-fold cross-validation for a more robust evaluation.
- Step 2: Normalize/scale features appropriately
  - Fit the scaler on the training data, then transform the validation and test data.
- Step 3: Start with a simple baseline model
  - Example: linear regression or logistic regression without heavy regularization.
- Step 4: Add regularization and/or simpler model constraints
  - Try Ridge/Lasso (L2/L1) and adjust strength with cross-validation.
- Step 5: Use cross-validation to choose hyperparameters
  - If you’re using a neural network or boosted trees, use a CV strategy suitable for the data and compute average validation performance.
- Step 6: Use early stopping if training iteratively
  - For neural nets or gradient-boosted methods, monitor validation loss and stop early.
- Step 7: After settling on a model, evaluate on the test set
  - This gives an unbiased estimate of real-world performance.
- Step 8: If performance is still lacking
  - Collect more data, engineer better features, reduce dimensionality, or try a different model family.
- Model-type tips
  - Linear models: prefer L2 and even feature selection via L1 if you have many features.
  - Decision trees/ensembles: control depth, learning rate, and use early stopping with validation; consider regularization parameters in boosting or random forests.
  - Neural networks: use dropout, weight decay, data augmentation (especially for images or time-series), and early stopping.

Quick practical code snippets (illustrative)
- Ridge regression with a train/validation split:
  - from sklearn.model_selection import train_test_split
  - from sklearn.linear_model import Ridge
  - from sklearn.metrics import mean_squared_error
  - X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
  - model = Ridge(alpha=1.0)  # regularization strength
  - model.fit(X_train, y_train)
  - pred = model.predict(X_val)
  - mse = mean_squared_error(y_val, pred)

- Cross-validation to pick alpha (regularization strength):
  - from sklearn.linear_model import Ridge
  - from sklearn.model_selection import RidgeCV
  - alphas = [0.1, 1.0, 10.0, 100.0]
  - ridgecv = RidgeCV(alphas=alphas, cv=5)
  - ridgecv.fit(X, y)
  - best_alpha = ridgecv.alpha_

- Early stopping in a neural network (Keras-like pseudocode):
  - from keras.callbacks import EarlyStopping
  - model.fit(X_train, y_train, validation_data=(X_val, y_val),
              epochs=100, callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)])

- Simple cross-validation score example:
  - from sklearn.model_selection import cross_val_score
  - scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')
  - mean_score = scores.mean()

- Important data leakage reminder
  - Always fit scalers/transformations on the training data only, then apply to validation/test:
  - scaler.fit(X_train)
  - X_train_scaled = scaler.transform(X_train)
  - X_val_scaled = scaler.transform(X_val)

If you want, tell me your data type (tabular, text, images) and your current model, and I’ll tailor a concrete plan with specific hyperparameters, cross-validation strategy, and sample code.

In [16]:
# Reusing Llama 3.2 to answer, with streaming
stream = ollama.chat.completions.create(
    model=MODEL_LLAMA,
    messages=messages,
    stream=True
)

response = ""
display_handle = display(Markdown(""), display_id=True)

for chunk in stream:
    response += chunk.choices[0].delta.content or ""
    update_display(
        Markdown(response),
        display_id=display_handle.display_id
    )

Overfitting Prevention: A Step-by-Step Guide
=============================================

**Why does overfitting happen?**

Overfitting occurs when a model is too complex and learns the noise in the training data. This results in the model performing exceptionally well on the training data but struggles to generalize to new, unseen data.

Imagine trying to fit a curve to a small dataset of noisy points. The model may fit the curve perfectly, but when you try to predict the curve on a larger, noisy dataset, it will not perform well.

**Why use both training and validation data?**

Using both training and validation data helps prevent overfitting. The training data is used to train the model, while the validation data is used to evaluate the model's performance during training.

Think of the training data as the "master recipe" and the validation data as the "taste test." You train the model using the training data, but you also check its performance on the validation data to make sure it's not overfitting.

Here's an example:

```python
# Import necessary libraries
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Load the iris dataset
iris = load_iris()

# Split the dataset into training and validation sets (10% for validation)
X_train, X_val, y_train, y_val = train_test_split(iris.data, iris.target, test_size=0.1, random_state=42)
```

**How does regularization help?**

Regularization is a technique that adds a penalty term to the loss function to discourage large weights. This helps prevent overfitting by forcing the model to use the most relevant features and not just memorizing the training data.

There are two types of regularization:

*   **L1 Regularization (Lasso):** Adds a term that decreases as the magnitude of the weights increases.
*   **L2 Regularization (Ridge):** Adds a term that decreases as the square of the magnitude of the weights increases.

Here's an example using L1 Regularization:

```python
# Import necessary libraries
from sklearn.linear_model import Lasso
from sklearn.datasets import load_iris

# Load the iris dataset
iris = load_iris()

# Create a Lasso model
model = Lasso(alpha=0.1)

# Train the model using the training data
model.fit(iris.data, iris.target)
```

**How does cross-validation help?**

Cross-validation is a technique that splits the data into smaller sets and evaluates the model's performance on each set independently. This helps prevent overfitting by ensuring that the model is generalizable to unseen data.

There are several types of cross-validation:

*   **K-Fold Cross-Validation:** Splits the data into k folds, and evaluates the model's performance on each fold.
*   **Stratified K-Fold Cross-Validation:** Splits the data into k folds while maintaining the class balance.

Here's an example using 5-Fold Cross-Validation:

```python
# Import necessary libraries
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Create a Random Forest model
model = RandomForestClassifier()

# Create a K-Fold object
 kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Iterate over the folds
for train_index, val_index in kf.split(iris.data):
    X_train, X_val = iris.data[train_index], iris.data[val_index]
    y_train, y_val = iris.target[train_index], iris.target[val_index]

    # Train the model using the training data
    model.fit(X_train, y_train)

    # Evaluate the model's performance on the validation data
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    print("Accuracy:", accuracy)
```

**How does early stopping help?**

Early stopping is a technique that stops training the model when its performance on the validation data starts to degrade. This helps prevent overfitting by stopping the training process before the model becomes too specialized to the training data.

Here's an example using early stopping:

```python
# Import necessary libraries
from sklearn.model_selection import EarlyStopping
from sklearn.ensemble import RandomForestClassifier

# Create a Random Forest model
model = RandomForestClassifier()

# Create an EarlyStopping object
early_stopping = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)

# Train the model using the training data with early stopping
model.fit(X_train, y_train, epochs=100, validation_data=(X_val, y_val), callbacks=[early_stopping])

# Print the trained model's performance on the validation data
y_pred = model.predict(X_val)
accuracy = accuracy_score(y_val, y_pred)
print("Accuracy:", accuracy)
```

By following these techniques, you can prevent overfitting and improve the generalization performance of your machine learning models.

# Week 2 Final Exercise - AI Technical Tutor

The Personalized Technical Tutor is now upgraded into a complete AI application.

The user can select an area of expertise, choose between different AI models and receive streaming responses through a Gradio interface.

In [54]:
# Import Gradio and json
import gradio as gr
import json

## Expertise

The user can select the technical area they want help with.

The selected expertise will be added to the System Prompt so the tutor can provide a more relevant explanation.

In [37]:
# Expertise

expertise_options = [
    "Python",
    "Data Science",
    "Machine Learning",
    "Generative AI",
    "SQL",
    "Data Structures and Algorithms"
]

In [38]:
# Models: The user can choose which AI model should answer the technical question.

model_options = ["GPT-5-nano","Llama 3.2"]

## System Prompt

The System Prompt will change based on the selected area of expertise.

This helps the tutor answer as an expert in the selected technical topic.

In [39]:
# System Prompt

def get_system_prompt(expertise):
    return f"""
You are a technical tutor with expertise in {expertise}.

Explain technical concepts in simple and clear steps.

Assume familiarity with Python, statistics, data analysis and machine learning.

When helpful:
- Explain the main idea first
- Break the answer into simple steps
- Use practical examples
- Explain formulas and calculations clearly
"""

In [40]:
# Technical Tutor

def tutor(message, history, expertise, model):

    system_prompt = get_system_prompt(expertise)

    messages = [{"role": "system", "content": system_prompt}]

    for item in history:
        messages.append({
            "role": item["role"],
            "content": item["content"]
        })

    messages.append({
        "role": "user",
        "content": message
    })

    if model == "GPT-5-nano":
        client = openai
        model_name = MODEL_GPT

    else:
        client = ollama
        model_name = MODEL_LLAMA

    stream = client.chat.completions.create(
        model=model_name,
        messages=messages,
        stream=True
    )

    response = ""

    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response

## Streaming

The tutor sends the question, conversation history and selected expertise to the chosen AI model.

The answer is streamed back gradually instead of waiting for the complete response.

The same function works with both GPT-5-nano and Llama 3.2.

## Technical Tutor Interface

Now we'll create a Gradio interface where the user can select the expertise, choose the model and ask technical questions.

In [55]:
# Gradio Interface

demo = gr.ChatInterface(
    fn=tutor,
    additional_inputs=[
        gr.Dropdown(
            choices=expertise_options,
            value="Machine Learning",
            label="Expertise"
        ),
        gr.Dropdown(
            choices=model_options,
            value="GPT-5-nano",
            label="Model"
        )
    ],
    type="messages",
    title="AI Technical Tutor",
    description="Choose an area of expertise and an AI model, then ask a technical question."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7886
* To create a public link, set `share=True` in `launch()`.


## Testing the Technical Tutor

The Technical Tutor can now:

- Answer technical questions
- Use conversation history
- Change its area of expertise
- Switch between GPT-5-nano and Llama 3.2
- Stream the response in the Gradio interface

## Adding Tool Calling

Now we'll extend our Technical Tutor with a tool that recommends learning resources.

In [56]:
# Learning Resources

learning_resources = {
    "python": "Python Official Tutorial",
    "sql": "SQLBolt Interactive SQL Lessons",
    "machine learning": "Scikit-learn User Guide",
    "generative ai": "Hugging Face LLM Course",
    "data structures and algorithms": "LeetCode Practice Problems"
}

### Learning Resources

This dictionary stores a recommended learning resource for each technical topic.

In [57]:
def get_learning_resource(topic):
    print(f"RESOURCE TOOL CALLED: Getting resource for {topic}", flush=True)

    resource = learning_resources.get(topic.lower())

    if resource:
        return f"Recommended resource for {topic}: {resource}"

    return f"No learning resource available for {topic}"

In [58]:
get_learning_resource("Python")

RESOURCE TOOL CALLED: Getting resource for Python


'Recommended resource for Python: Python Official Tutorial'

## Learning Resource Tool

Now we'll define the learning resource function as a tool that the AI can call.

In [59]:
learning_resource_function = {
    "name": "get_learning_resource",
    "description": "Get a recommended learning resource for a technical topic.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "The technical topic, for example Python or Machine Learning"
            }
        },
        "required": ["topic"],
        "additionalProperties": False
    }
}

tools = [
    {
        "type": "function",
        "function": learning_resource_function
    }
]

In [60]:
# This function handles tool requests from the AI and returns the result back to the model.
def handle_tool_calls(message):
    responses = []

    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_learning_resource":
            arguments = json.loads(tool_call.function.arguments)
            topic = arguments.get("topic")

            result = get_learning_resource(topic)

            responses.append({
                "role": "tool",
                "content": result,
                "tool_call_id": tool_call.id
            })

    return responses

In [61]:
# Adding Audio Input
def transcribe_audio(audio_file):

    if audio_file is None:
        return ""

    with open(audio_file, "rb") as audio:
        transcription = openai.audio.transcriptions.create(
            model="whisper-1",
            file=audio
        )

    return transcription.text

### Speech-to-Text

This function converts the user's recorded audio into text using OpenAI's speech-to-text model.

The transcribed text can then be sent to our existing Technical Tutor.

In [62]:
audio_input = gr.Audio(
    sources=["microphone"],
    type="filepath",
    label="Record a Question"
)

transcription_output = gr.Textbox(
    label="Transcribed Question"
)

audio_demo = gr.Interface(
    fn=transcribe_audio,
    inputs=audio_input,
    outputs=transcription_output,
    title="Audio Input Test"
)

audio_demo.launch()

* Running on local URL:  http://127.0.0.1:7887
* To create a public link, set `share=True` in `launch()`.


## Connecting Audio to the Technical Tutor

Now we'll connect the audio transcription to our existing Technical Tutor.

The user can speak a question, convert it into text and send the question to the tutor.

In [63]:
def voice_question(audio_file):

    if audio_file is None:
        return ""

    question = transcribe_audio(audio_file)

    return question

In [64]:
with gr.Blocks() as demo:

    gr.Markdown("# AI Technical Tutor")
    gr.Markdown(
        "Ask a technical question by typing or using your voice."
    )

    chatbot = gr.Chatbot(type="messages")

    with gr.Row():

        expertise = gr.Dropdown(
            choices=expertise_options,
            value="Machine Learning",
            label="Expertise"
        )

        model = gr.Dropdown(
            choices=model_options,
            value="GPT-5-nano",
            label="Model"
        )

    message = gr.Textbox(
        label="Question",
        placeholder="Ask a technical question..."
    )

    audio = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Record a Question"
    )

    audio.change(
        fn=voice_question,
        inputs=audio,
        outputs=message
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7888
* To create a public link, set `share=True` in `launch()`.


In [65]:
## Adding Audio Output
def talker(message):

    audio_file = "response.mp3"

    with openai.audio.speech.with_streaming_response.create(
        model="tts-1",
        voice="onyx",
        input=message
    ) as response:
        response.stream_to_file(audio_file)

    return audio_file

In [66]:
gr.Audio(
    value="response.mp3",
    autoplay=True,
    label="Tutor Audio Response"
)

In [67]:
def tutor(message, history, expertise, model):
    system_prompt = get_system_prompt(expertise)
    messages = [{"role": "system", "content": system_prompt}]

    for item in history:
        messages.append({"role": item["role"],"content": item["content"]})

    messages.append({
        "role": "user",
        "content": message
    })

    if model == "GPT-5-nano":

        response = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            tools=tools
        )

        if response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            responses = handle_tool_calls(assistant_message)

            messages.append(assistant_message)
            messages.extend(responses)

        stream = openai.chat.completions.create(
            model=MODEL_GPT,
            messages=messages,
            tools=tools,
            stream=True
        )

    else:

        stream = ollama.chat.completions.create(
            model=MODEL_LLAMA,
            messages=messages,
            stream=True
        )

    answer = ""

    for chunk in stream:
        answer += chunk.choices[0].delta.content or ""
        yield answer

In [ ]:
with gr.Blocks() as demo:

    gr.Markdown("# AI Technical Tutor")
    gr.Markdown(
        "Ask a technical question by typing or using your voice."
    )

    chatbot = gr.Chatbot(type="messages")

    with gr.Row():
        expertise = gr.Dropdown(
            choices=expertise_options,
            value="Machine Learning",
            label="Expertise"
        )

        model = gr.Dropdown(
            choices=model_options,
            value="GPT-5-nano",
            label="Model"
        )

    message = gr.Textbox(
        label="Question",
        placeholder="Ask a technical question..."
    )

    audio_input = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="Record a Question"
    )

    audio_output = gr.Audio(
        label="Tutor Audio Response",
        autoplay=True
    )

    audio_input.change(
        fn=voice_question,
        inputs=audio_input,
        outputs=message
    )

    def respond(user_message, history, expertise, model):

        history = history + [
            {"role": "user", "content": user_message}
        ]

        yield "", history, None

        answer = ""

        for partial_response in tutor(
            user_message,
            history[:-1],
            expertise,
            model
        ):

            answer = partial_response

            updated_history = history + [
                {"role": "assistant", "content": answer}
            ]

            yield "", updated_history, None

        audio_file = talker(answer)

        yield "", updated_history, audio_file

    message.submit(
        fn=respond,
        inputs=[
            message,
            chatbot,
            expertise,
            model
        ],
        outputs=[
            message,
            chatbot,
            audio_output
        ]
    )

demo.launch()

* Running on local URL:  http://127.0.0.1:7889
* To create a public link, set `share=True` in `launch()`.


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\uvicorn\protocols\http\httptools_impl.py", line 409, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\uvicorn\middleware\proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\fastapi\applications.py", line 1134, in __call__
    await super().__call__(scope, receive, send)
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\site-packages\starlette\applications.py", line 113, in __call__
    await self.middleware_stack(scope, receive, send)
  File "c:\Users\Yesh Damania\Projects\llm_engineering\.venv\Lib\si

RESOURCE TOOL CALLED: Getting resource for Python
